# 03 — TRACe Judge Validation (the hard half) — Group 16

**Goal:** before we trust the LLM *judge* to score our own RAG pipeline, measure how well its TRACe scores agree with RAGBench's shipped **reference** scores — and use that to **pick which open-source judge model** to commit to. This is the **trust gate** for the entire Task-1 results matrix: every number in that matrix is only as reliable as the judge that produced it.

**How this differs from notebook 02.** Notebook 02 validated the *arithmetic* half — given gold sentence labels, our formulas reproduced the reference scores to **RMSE ~ 0**. That was a unit test. This notebook validates the *judge* half — the LLM that *produces* those labels (which sentences are relevant `R`, which were utilized `U`, whether the answer is supported).

> WARNING: **Do not expect RMSE = 0 here.** The judge makes the same *subjective* calls a human / GPT-4 annotator makes, so it will never match the reference exactly. Success is **good agreement**, not perfect reproduction — and, crucially, **the best-agreeing model among the candidates**. RAGBench's own GPT-4 annotator reached ~0.93–0.95 agreement with humans; the paper also found **relevance is the hardest metric to predict** (highest error), so expect relevance RMSE to be the worst of the three.

**What we measure** (judge vs reference, via `src/evaluator/judge_validate.py`):
- 3 fraction metrics (relevance / utilization / completeness) -> **RMSE** + mean absolute error (lower = closer).
- adherence (yes / no) -> **accuracy**.

**The method.** We feed the judge each example's *own* keyed sentences (the same `documents_sentences` / `response_sentences` GPT-4 saw), let it derive `R` / `U` / support, map those to TRACe scores with the **validated** `scores_from_label` adapter, and compare to the reference. Same inputs, compare outputs.

> Runs on **Colab GPU** (the judge is a real LLM). Budget ~20–40 min for a small run; scale `N` once you've chosen a model.

In [ ]:
# Colab setup: clone the repo and install the judge's dependencies.
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/abhisagar123/CapstoneRAG.git
    os.chdir("CapstoneRAG")
sys.path.insert(0, os.getcwd())

# datasets       = RAGBench loader
# transformers   = run the judge model        accelerate = device placement
# bitsandbytes   = 4-bit quantization (lets a 7-8B model fit a free-Colab T4)
!pip install -q datasets transformers accelerate bitsandbytes

## (optional) Hugging Face token

Set this if you hit download rate-limits, or if you switch to a **gated** model (Llama-3) which won't download without it. **Qwen2.5 is ungated — you can skip this.**

**Safe setup:** add the token in Colab's **🔑 Secrets** panel (left sidebar) as `HF_TOKEN`, toggle *Notebook access* on. The cell below reads it from there — **never paste the token into a cell** (this repo is public; a committed token is a leaked credential).

In [ ]:
# Load HF_TOKEN from Colab Secrets if present. Optional + safe:
#  - no token set  -> silently continue (fine for ungated models like Qwen2.5)
#  - not on Colab   -> silently continue (the import only exists on Colab)
# Never hardcode the token here: this repo is public.
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        import os
        os.environ['HF_TOKEN'] = _tok
        print('HF_TOKEN loaded from Colab Secrets.')
    else:
        print('No HF_TOKEN secret set — continuing unauthenticated (ok for ungated models).')
except Exception:
    print('Not on Colab (or Secrets unavailable) — continuing without HF_TOKEN.')

## 1. Choose the judge model

The judge **defines ground truth**, so it should be a strong instruct model — but it must fit your Colab GPU. Set `MODEL` below. Approximate footprints (4-bit):

| Model | Params | 4-bit VRAM | Fits free T4 (16 GB)? | License / gating |
|---|---|---|---|---|
| `Qwen/Qwen2.5-7B-Instruct` *(default)* | 7B | ~6 GB | yes | Apache-2.0 — **ungated**, no token |
| `meta-llama/Meta-Llama-3-8B-Instruct` | 8B | ~6 GB | yes | **Gated** — accept Meta license + set `HF_TOKEN` |
| `Qwen/Qwen2.5-32B-Instruct` | 32B | ~20 GB | no — needs L4 / A100 | Apache-2.0 — ungated |
| `meta-llama/Meta-Llama-3-70B-Instruct` | 70B | ~38 GB | no — needs A100-40GB (Pro) | **Gated** — license + `HF_TOKEN` |

**Default = `Qwen2.5-7B-Instruct`**: strong, ungated (zero friction), and fits the free T4 in 4-bit. If you have Colab Pro and want your preferred **Llama-3-70B**, set `MODEL` to it, accept the license on its HF page, and add your `HF_TOKEN` (Colab *Secrets*).

This notebook validates **one model per run** and appends to `results/judge_validation.csv`. To compare another model: change `MODEL`, **restart the runtime** (frees VRAM), and re-run — the comparison table at the end grows automatically.

In [ ]:
# --- Edit these, then run top-to-bottom ---
MODEL          = "Qwen/Qwen2.5-7B-Instruct"   # see the table above
LOAD_IN_4BIT   = True      # True on T4 (fits 7-8B); False on A100/L4 for full precision
MAX_NEW_TOKENS = 1536      # the Appendix-7.4 JSON is verbose; raise for long contexts (CUAD)
N              = 15        # examples PER config (judge is slow; start small, scale later)

# One representative config per domain. Trim/extend as you like.
# (cuad = long legal contracts -> slowest; drop it for a quick first pass.)
CONFIGS_TO_VALIDATE = [
    ("Biomedical",      "covidqa"),
    ("GenKnowledge",    "hotpotqa"),
    ("Legal",           "cuad"),
    ("CustomerSupport", "emanual"),
    ("Finance",         "tatqa"),
]

# Gated models (Llama) need a token — set it in the HF_TOKEN cell above (Colab Secrets).

In [ ]:
import src                         # registers the light components
from src.judge import load_judges
load_judges()                       # registers the heavy HuggingFaceJudge ("hf")

from src.registry import build
# Build via the registry — the same swap-by-config mechanism the pipeline uses.
judge = build("judge", "hf", {
    "model": MODEL, "load_in_4bit": LOAD_IN_4BIT, "max_new_tokens": MAX_NEW_TOKENS,
})
print(f"Judge ready: {MODEL}  (4-bit={LOAD_IN_4BIT})")

## 2. Smoke test — does the judge emit parseable labels?

Before the (slow) full loop, run the judge on **one** labelled example. OSS models are messier than GPT-4 at clean JSON; this fails fast if the output can't be parsed, instead of wasting 20 minutes.

In [ ]:
from datasets import load_dataset
from src.evaluator.validate import REPO
from src.evaluator.judge_validate import _keyed_from_example
from src.judge import scores_from_label

ex = load_dataset(REPO, CONFIGS_TO_VALIDATE[0][1], split="test")[0]
keyed = _keyed_from_example(ex)
label = judge.label(ex["question"], keyed)        # the LLM call + JSON parse
print("Parsed judge label — keys:", sorted(label))

ours = scores_from_label(keyed, label)
print("\n              judge      reference")
for m, fld in [("relevance","relevance_score"), ("utilization","utilization_score"),
               ("completeness","completeness_score"), ("adherence","adherence_score")]:
    print(f"{m:13} {str(ours[m]):10} {ex[fld]}")
print("\nJudge produced parseable labels. (One example — agreement is measured at scale below.)")

## 3. Validate across domains

Run the judge on `N` examples of each chosen config and compare to the reference. Results append to `results/judge_validation.csv` (resumable — already-done `(model, config)` pairs are skipped, so a Colab disconnect won't lose finished configs).

In [ ]:
import os, csv
import pandas as pd
from src.evaluator.judge_validate import validate_judge

OUT  = "results/judge_validation.csv"
COLS = ["model","domain","config","n","n_scored","n_failed","relevance_rmse","relevance_mae",
        "utilization_rmse","utilization_mae","completeness_rmse","completeness_mae",
        "adherence_acc"]

def flatten(report, model, domain):
    r = {"model": model, "domain": domain, "config": report["config"],
         "n": report["n"], "n_scored": report["n_scored"], "n_failed": report["n_failed"]}
    for m in ["relevance","utilization","completeness"]:
        r[f"{m}_rmse"] = report[m]["rmse"]
        r[f"{m}_mae"]  = report[m]["mean_abs_err"]
    r["adherence_acc"] = report["adherence"]["accuracy"]
    return r

done = set()
if os.path.exists(OUT):
    _d = pd.read_csv(OUT)
    done = set(zip(_d["model"], _d["config"]))   # skip (model, config) already validated

write_header = not os.path.exists(OUT)
with open(OUT, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLS)
    if write_header:
        w.writeheader()
    for domain, config in CONFIGS_TO_VALIDATE:
        if (MODEL, config) in done:
            print(f"skip (done): {MODEL} / {config}"); continue
        print(f"judging {N}x {domain}/{config} ...", flush=True)
        report = validate_judge(judge, config, n=N, split="test")
        w.writerow(flatten(report, MODEL, domain)); f.flush()   # persist per-config (Colab-safe)
        print(f"   -> rel_rmse={report['relevance']['rmse']:.3f}  adh_acc={report['adherence']['accuracy']:.2f}"
              f"  (scored {report['n_scored']}/{report['n']}, {report['n_failed']} unparseable)")
print("done.")

## 4. Verdict — agreement per model, and which judge to pick

In [ ]:
import pandas as pd
df = pd.read_csv(OUT)

# Per-model mean agreement across the validated configs.
summary = df.groupby("model").agg(
    configs        = ("config", "nunique"),
    examples       = ("n", "sum"),
    relevance_rmse = ("relevance_rmse", "mean"),
    utilization_rmse = ("utilization_rmse", "mean"),
    completeness_rmse = ("completeness_rmse", "mean"),
    adherence_acc  = ("adherence_acc", "mean"),
).round(3)

print("Mean judge<->reference agreement (lower RMSE = closer; adherence = accuracy):\n")
print(summary)
print("\nPer-config detail:")
print(df.round(3).to_string(index=False))
print("\n-> Pick the model with the best blend of LOW RMSE (esp. relevance — the hardest)")
print("   and HIGH adherence accuracy. Record it as THE judge for the results matrix.")

## 5. How to read this + honest caveats

**Picking the judge.** Lower fraction-RMSE + higher adherence accuracy = better agreement with the reference. There is **no RMSE ~ 0 pass/fail** here (unlike notebook 02) — the judge is subjective. The decision is **relative**: run 2–3 candidates, pick the best, commit to it for the whole matrix, and **report its agreement numbers** (they bound how much to trust the matrix).

- **Relevance is hardest** (RAGBench paper — highest RMSE). Don't be alarmed if it trails utilization / completeness.
- **Adherence** is a single bool from the judge's `overall_supported`; small samples make accuracy lumpy.
- **Sampled & small `N`** — this *picks* a model; it is not the final agreement report. Re-run the winner at higher `N` for the number you cite.
- **Unparseable JSON is skipped, not fatal.** OSS judges sometimes emit malformed JSON (e.g. an unescaped quote inside an explanation). The judge retries once **with sampling** (a greedy retry would repeat the same bad text), and a salvage pass repairs unescaped inner quotes. If it still won't parse, that **one example is skipped and counted** in `n_failed` — it never kills the whole config. A high `n_failed` is itself a judge-quality signal (prefer a model that parses reliably).
- **No caching yet** — re-running re-judges (cost / time). Resumability is per-`(model, config)`, not per-example; a mid-config disconnect re-does that config. (Caching is a tracked TODO.)
- **Gated models** (Llama) need license acceptance + `HF_TOKEN`; **Qwen2.5** is ungated.
- **`MAX_NEW_TOKENS`** may truncate the JSON on very long contexts (CUAD) -> a parse error. Raise it, or lower `N` for legal.